# 04. Properties

**Tier:** Intermediate. **Why you'd use this:** A temperature stored in Celsius should expose Fahrenheit as if it were a field. A property computes on read without changing the call site to a method call.

**After this lesson you can:** write a `@property`; use `@classmethod` and `@staticmethod` for the right first argument.
About 25 minutes. **Uses:** [Classes](01-classes-lesson.ipynb), [Dataclasses](02-dataclasses-lesson.ipynb)


## Quick review

1. What does `super()` follow?
2. Why print `__mro__`?

<details><summary>Answers</summary>

The method resolution order. To see which class is tried next on lookup.
</details>


## The idea

A property turns a method into an attribute-style read. It is a descriptor on the class. Classmethods receive the class. Staticmethods receive neither instance nor class. Choose the binding that matches the job.

<details><summary>Explain more</summary>

`celsius.fahrenheit` calls the getter. No parentheses. A setter can validate writes when you add one.
</details>


## See it work

Expose Fahrenheit from Celsius degrees.


In [1]:
class Celsius:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def fahrenheit(self):
        # 1. Computed on read like an attribute.
        return self.degrees * 9 / 5 + 32


print(Celsius(0).fahrenheit)
print(Celsius(100).fahrenheit)


32.0
212.0


## Check yourself

Answer these before you run the cell.

1. Do you call a property with parentheses?
2. What does a classmethod receive as its first argument?


In [2]:
class Celsius:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def fahrenheit(self):
        return self.degrees * 9 / 5 + 32

    @classmethod
    def from_fahrenheit(cls, value):
        return cls((value - 32) * 5 / 9)


print(Celsius(0).fahrenheit)
print(Celsius.from_fahrenheit(32).degrees)


32.0
0.0


<details><summary>Answers</summary>

No. The class.
</details>


## Why it works this way

A `property` is a descriptor: an object that customizes get and set when it sits on a class. Looking up `celsius.fahrenheit` calls the getter with the instance. That is why a method can look like a field. Attribute lookup finds the property on the class before a missing instance attribute would fail. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

`@classmethod` receives the class as the first argument, useful for alternate constructors. `@staticmethod` receives neither class nor instance; it is a namespaced function. All three use decorator syntax you first met with dataclasses. Underneath, each builds a descriptor or wraps a function onto the class.

Cost of a property get is a normal function call. The design reason is a stable public attribute that can later grow checks without changing call sites from `obj.x` to `obj.x()`. Start with a plain attribute. Promote to a property when you need computation or validation while keeping attribute syntax.

Evidence: reading `.fahrenheit` returns a computed float, and assigning to a read-only property raises. Classmethods and staticmethods show different first-argument binding when inspected or called on the class.


Properties also document intent. A bare attribute says "this is data." A property says "this may compute or validate." Call sites stay stable either way, which is why teams promote fields to properties during growth instead of renaming everything to methods. Prefer a method when the operation feels like an action with arguments. Prefer a property when it feels like a derived field. Measuring nothing exotic is required here: the mechanism is lookup and descriptors, and the cost is one extra call on access.


In [3]:
class Celsius:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def fahrenheit(self):
        return self.degrees * 9 / 5 + 32

    @classmethod
    def from_fahrenheit(cls, value):
        return cls((value - 32) * 5 / 9)

    @staticmethod
    def label():
        return "C"


zero = Celsius(0)
print(zero.fahrenheit)
print(Celsius.from_fahrenheit(32).degrees)
print(Celsius.label())


32.0
0.0
C


## When to use it

Use a property when call sites should keep attribute syntax while you compute or validate. Use a classmethod for alternate constructors. Use a staticmethod for a pure function that belongs in the class namespace. Don't make everything a property; plain attributes are fine until you need behaviour on access.


## Common mistake

Treating a property like a method and calling it with parentheses, or forgetting that a read-only property rejects assignment.


In [4]:
class Celsius:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def fahrenheit(self):
        return self.degrees * 9 / 5 + 32


print(Celsius(0).fahrenheit())


TypeError: 'float' object is not callable

Read it as an attribute. Add a setter only when writes should be allowed.


In [5]:
class Celsius:
    def __init__(self, degrees):
        self.degrees = degrees

    @property
    def fahrenheit(self):
        return self.degrees * 9 / 5 + 32


print(Celsius(0).fahrenheit)


32.0


## Try it

Add a setter for `fahrenheit` that updates `degrees`. Which decorator stacks under `@property`, and what invariant should the setter preserve?


## Recap

- A property is a descriptor that computes on attribute access.
- Classmethods get the class; staticmethods get neither.
- Keep attribute syntax stable when promoting a field to a property.

**Next:** [practice](04-properties-practice.ipynb)
